User Input
      ↓
ChatPromptTemplate
      ↓
Formatted Prompt
      ↓
ChatOpenAI (GPT-4o)
      ↓
Model Response
      ↓
LLMChain returns output

In [2]:
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

True

In [3]:
TEMPLATE  = """"
Interpreret the text and Evaluate the text.
sentiment: is the text positive, negative or neutral?
subject : what subject is the text about?
price : how much did the customer paid for the product? use the price mentioned in the text, if not mentioned then use 0.

format the output in json format with keys 
sentiment 
subject
 price.

 text: {input}
"""

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_classic import LLMChain

llm = ChatOpenAI(model_name="gpt-4o", temperature=0.9)

prompt_template = ChatPromptTemplate.from_template(template = TEMPLATE)
chain= LLMChain(llm=llm, prompt=prompt_template)    
chain.run(input="I bought a new phone for $500 and I love it!")



/var/folders/m9/b1vwyk8j689_dblx3cn469_c0000gn/T/ipykernel_16343/2245256812.py:9: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  chain.run(input="I bought a new phone for $500 and I love it!")


'```json\n{\n  "sentiment": "positive",\n  "subject": "phone purchase",\n  "price": 500\n}\n```'

Building Response Schemas

In [19]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

class ReviewAnalysis(BaseModel):
    sentiment: str = Field(..., description="The sentiment of the text: positive, negative, or neutral.")
    subject: str = Field(..., description="The subject of the text.")
    price: float = Field(..., description="The price mentioned in the text. If not mentioned, use 0.")

parser = PydanticOutputParser(pydantic_object=ReviewAnalysis)    


In [20]:
format_instructions = parser.get_format_instructions()

print(format_instructions)

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"sentiment": {"description": "The sentiment of the text: positive, negative, or neutral.", "title": "Sentiment", "type": "string"}, "subject": {"description": "The subject of the text.", "title": "Subject", "type": "string"}, "price": {"description": "The price mentioned in the text. If not mentioned, use 0.", "title": "Price", "type": "number"}}, "required": ["sentiment", "subject", "price"]}
```


In [25]:
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate


prompt = ChatPromptTemplate(
    messages=[
    SystemMessagePromptTemplate.from_template(
    """
    Interpreret the text and Evaluate the text.
    sentiment: is the text positive, negative or neutral?,
    subject : what subject is the text about?,
    price : how much did the customer paid for the product? use the price mentioned in the text, if not mentioned then use 0.,
    Do not put any text outside of the json format. Only return the json format.,
    text: {input}
    format_instructions: {format_instructions}
    """
    )
],
    input_variables=["input"],
    partial_variables={"format_instructions": format_instructions}
    )

In [27]:
input_text = "I bought a new phone for $500 and I love it!"
output = llm.invoke(prompt.format(input=input_text))
print(output)

content='```json\n{"sentiment": "positive", "subject": "phone purchase", "price": 500}\n```' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 342, 'total_tokens': 366, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_e8ddfc8c33', 'id': 'chatcmpl-DssH1KO2EA8AqlDRGYHaI27uGxka8', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019ee5bb-2e4f-74e2-b104-df57d30529ba-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 342, 'output_tokens': 24, 'total_tokens': 366, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [29]:
json_output = parser.parse(output.content)
print(json_output)

sentiment='positive' subject='phone purchase' price=500.0
